# K-Means Clustering: From Theory to Implementation

A comprehensive guide to understanding, implementing, and optimizing K-Means clustering.

---

## Table of Contents

1. [Theory Section](#1-theory-section)
2. [Implementation from Scratch](#2-implementation-from-scratch)
3. [Training & Optimization](#3-training--optimization)
4. [Diagnostics & Evaluation](#4-diagnostics--evaluation)
5. [Visualizations](#5-visualizations)
6. [Use Cases & Guidelines](#6-use-cases--guidelines)
7. [Comparison with sklearn](#7-comparison-with-sklearn)

---

## 1. Theory Section

### What is K-Means Clustering?

K-Means is an unsupervised learning algorithm that partitions n observations into k clusters, where each observation belongs to the cluster with the nearest centroid (cluster center).

### Lloyd's Algorithm

The standard K-Means algorithm, known as Lloyd's algorithm, consists of the following steps:

1. **Initialization**: Select k initial centroids (randomly or using k-means++)
2. **Assignment Step**: Assign each data point to the nearest centroid
3. **Update Step**: Recalculate centroids as the mean of all points assigned to each cluster
4. **Repeat**: Continue steps 2-3 until convergence

**Mathematical Formulation:**

The objective is to minimize the Within-Cluster Sum of Squares (WCSS):

$$J = \sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2$$

Where:
- $k$ = number of clusters
- $C_i$ = set of points in cluster $i$
- $\mu_i$ = centroid of cluster $i$

### Centroid Initialization Methods

#### Random Initialization
- Randomly select k data points as initial centroids
- Simple but can lead to poor convergence or local minima
- Results vary significantly with different random seeds

#### K-Means++ Initialization
A smarter initialization that spreads out initial centroids:

1. Choose the first centroid uniformly at random from data points
2. For each data point x, compute $D(x)$ = distance to nearest existing centroid
3. Choose next centroid with probability proportional to $D(x)^2$
4. Repeat steps 2-3 until k centroids are chosen

**Benefits of K-Means++:**
- More consistent results
- Faster convergence
- Provably better bounds on the final objective

### Convergence Criteria

The algorithm stops when:
1. **Centroid movement**: Centroids move less than a tolerance threshold
2. **Assignment stability**: No points change cluster membership
3. **Maximum iterations**: Reached the iteration limit
4. **Inertia change**: WCSS improvement below threshold

### Choosing K: The Number of Clusters

#### Elbow Method
- Plot WCSS (inertia) vs. number of clusters
- Look for the "elbow" point where adding more clusters gives diminishing returns
- Subjective but widely used

#### Silhouette Score
For each sample:
$$s = \frac{b - a}{\max(a, b)}$$

Where:
- $a$ = mean intra-cluster distance
- $b$ = mean nearest-cluster distance

Score ranges from -1 to 1:
- 1: Sample is well-matched to its cluster
- 0: Sample is on the decision boundary
- -1: Sample might be assigned to wrong cluster

### Time and Space Complexity

| Aspect | Complexity |
|--------|------------|
| Time per iteration | O(n * k * d) |
| Total time | O(n * k * d * i) |
| Space | O(n * d + k * d) |

Where:
- n = number of samples
- k = number of clusters
- d = number of features
- i = number of iterations

**Note**: K-means++ initialization adds O(n * k * d) time but typically reduces total iterations needed.

---

## 2. Implementation from Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple, Literal

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
class KMeans:
    """
    K-Means clustering algorithm implementation from scratch.
    
    Parameters
    ----------
    n_clusters : int, default=8
        The number of clusters to form.
    init : {'k-means++', 'random'}, default='k-means++'
        Method for initialization of centroids.
    max_iter : int, default=300
        Maximum number of iterations.
    tol : float, default=1e-4
        Tolerance for declaring convergence based on centroid movement.
    n_init : int, default=10
        Number of times the algorithm will run with different centroid seeds.
    random_state : int or None, default=None
        Random seed for reproducibility.
    
    Attributes
    ----------
    cluster_centers_ : ndarray of shape (n_clusters, n_features)
        Coordinates of cluster centers.
    labels_ : ndarray of shape (n_samples,)
        Labels of each point.
    inertia_ : float
        Sum of squared distances to closest cluster center.
    n_iter_ : int
        Number of iterations run.
    """
    
    def __init__(
        self,
        n_clusters: int = 8,
        init: Literal['k-means++', 'random'] = 'k-means++',
        max_iter: int = 300,
        tol: float = 1e-4,
        n_init: int = 10,
        random_state: Optional[int] = None
    ):
        self.n_clusters = n_clusters
        self.init = init
        self.max_iter = max_iter
        self.tol = tol
        self.n_init = n_init
        self.random_state = random_state
        
        # Attributes set during fitting
        self.cluster_centers_ = None
        self.labels_ = None
        self.inertia_ = None
        self.n_iter_ = None
        self._centroid_history = None  # For visualization
    
    def _init_random(self, X: np.ndarray, rng: np.random.Generator) -> np.ndarray:
        """
        Initialize centroids by randomly selecting k data points.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Training data.
        rng : numpy.random.Generator
            Random number generator.
        
        Returns
        -------
        centroids : ndarray of shape (n_clusters, n_features)
            Initial centroid positions.
        """
        n_samples = X.shape[0]
        indices = rng.choice(n_samples, size=self.n_clusters, replace=False)
        return X[indices].copy()
    
    def _init_kmeans_plus_plus(self, X: np.ndarray, rng: np.random.Generator) -> np.ndarray:
        """
        Initialize centroids using k-means++ algorithm.
        
        The algorithm selects initial centroids that are well-spread out,
        leading to better convergence.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Training data.
        rng : numpy.random.Generator
            Random number generator.
        
        Returns
        -------
        centroids : ndarray of shape (n_clusters, n_features)
            Initial centroid positions.
        """
        n_samples, n_features = X.shape
        centroids = np.empty((self.n_clusters, n_features), dtype=X.dtype)
        
        # Step 1: Choose first centroid uniformly at random
        first_idx = rng.integers(0, n_samples)
        centroids[0] = X[first_idx]
        
        # Step 2-4: Choose remaining centroids with probability proportional to D(x)^2
        for k in range(1, self.n_clusters):
            # Compute squared distances to nearest existing centroid
            # Using broadcasting: (n_samples, 1, n_features) - (1, k, n_features)
            distances_sq = np.sum((X[:, np.newaxis, :] - centroids[np.newaxis, :k, :]) ** 2, axis=2)
            min_distances_sq = np.min(distances_sq, axis=1)
            
            # Convert to probabilities
            probabilities = min_distances_sq / min_distances_sq.sum()
            
            # Choose next centroid
            next_idx = rng.choice(n_samples, p=probabilities)
            centroids[k] = X[next_idx]
        
        return centroids
    
    def _compute_distances(self, X: np.ndarray, centroids: np.ndarray) -> np.ndarray:
        """
        Compute squared Euclidean distances from each point to each centroid.
        
        Uses the identity: ||x - y||^2 = ||x||^2 + ||y||^2 - 2*x.y
        for efficient computation.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Data points.
        centroids : ndarray of shape (n_clusters, n_features)
            Current centroid positions.
        
        Returns
        -------
        distances_sq : ndarray of shape (n_samples, n_clusters)
            Squared distances from each point to each centroid.
        """
        # ||x - c||^2 = ||x||^2 + ||c||^2 - 2*x.c
        X_sq = np.sum(X ** 2, axis=1, keepdims=True)  # (n_samples, 1)
        centroids_sq = np.sum(centroids ** 2, axis=1)  # (n_clusters,)
        cross_term = X @ centroids.T  # (n_samples, n_clusters)
        
        distances_sq = X_sq + centroids_sq - 2 * cross_term
        
        # Handle numerical precision issues
        np.maximum(distances_sq, 0, out=distances_sq)
        
        return distances_sq
    
    def _assign_clusters(self, X: np.ndarray, centroids: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Assign each data point to the nearest centroid.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Data points.
        centroids : ndarray of shape (n_clusters, n_features)
            Current centroid positions.
        
        Returns
        -------
        labels : ndarray of shape (n_samples,)
            Cluster assignment for each point.
        inertia : float
            Sum of squared distances to assigned centroids.
        """
        distances_sq = self._compute_distances(X, centroids)
        labels = np.argmin(distances_sq, axis=1)
        
        # Compute inertia (sum of squared distances to assigned centroids)
        inertia = np.sum(distances_sq[np.arange(len(labels)), labels])
        
        return labels, inertia
    
    def _update_centroids(self, X: np.ndarray, labels: np.ndarray) -> np.ndarray:
        """
        Update centroids as the mean of assigned points.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Data points.
        labels : ndarray of shape (n_samples,)
            Current cluster assignments.
        
        Returns
        -------
        new_centroids : ndarray of shape (n_clusters, n_features)
            Updated centroid positions.
        """
        n_features = X.shape[1]
        new_centroids = np.empty((self.n_clusters, n_features), dtype=X.dtype)
        
        for k in range(self.n_clusters):
            cluster_mask = labels == k
            if np.any(cluster_mask):
                new_centroids[k] = X[cluster_mask].mean(axis=0)
            else:
                # Handle empty cluster: reinitialize to a random point
                new_centroids[k] = X[np.random.randint(len(X))]
        
        return new_centroids
    
    def _single_run(
        self, 
        X: np.ndarray, 
        rng: np.random.Generator,
        track_history: bool = False
    ) -> Tuple[np.ndarray, np.ndarray, float, int, Optional[list]]:
        """
        Run K-Means algorithm once with given initialization.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Training data.
        rng : numpy.random.Generator
            Random number generator.
        track_history : bool, default=False
            Whether to track centroid history for visualization.
        
        Returns
        -------
        centroids : ndarray of shape (n_clusters, n_features)
            Final centroid positions.
        labels : ndarray of shape (n_samples,)
            Final cluster assignments.
        inertia : float
            Final inertia value.
        n_iter : int
            Number of iterations performed.
        history : list or None
            Centroid positions at each iteration if track_history=True.
        """
        # Initialize centroids
        if self.init == 'k-means++':
            centroids = self._init_kmeans_plus_plus(X, rng)
        else:
            centroids = self._init_random(X, rng)
        
        history = [centroids.copy()] if track_history else None
        
        # Main loop
        for iteration in range(self.max_iter):
            # Assignment step
            labels, inertia = self._assign_clusters(X, centroids)
            
            # Update step
            new_centroids = self._update_centroids(X, labels)
            
            if track_history:
                history.append(new_centroids.copy())
            
            # Check for convergence (centroid movement below tolerance)
            centroid_shift = np.sqrt(np.sum((new_centroids - centroids) ** 2, axis=1))
            if np.all(centroid_shift <= self.tol):
                centroids = new_centroids
                break
            
            centroids = new_centroids
        
        # Final assignment with converged centroids
        labels, inertia = self._assign_clusters(X, centroids)
        
        return centroids, labels, inertia, iteration + 1, history
    
    def fit(self, X: np.ndarray) -> 'KMeans':
        """
        Fit the K-Means model to the data.
        
        Runs the algorithm n_init times and keeps the best result
        (lowest inertia).
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Training data.
        
        Returns
        -------
        self : KMeans
            Fitted estimator.
        """
        X = np.asarray(X, dtype=np.float64)
        
        if X.shape[0] < self.n_clusters:
            raise ValueError(
                f"n_samples={X.shape[0]} should be >= n_clusters={self.n_clusters}"
            )
        
        # Set up random number generator
        rng = np.random.default_rng(self.random_state)
        
        best_inertia = np.inf
        best_centroids = None
        best_labels = None
        best_n_iter = None
        best_history = None
        
        # Run multiple times and keep best result
        for run in range(self.n_init):
            # Track history only for the first run (for visualization)
            track_history = (run == 0)
            
            centroids, labels, inertia, n_iter, history = self._single_run(
                X, rng, track_history=track_history
            )
            
            if inertia < best_inertia:
                best_inertia = inertia
                best_centroids = centroids
                best_labels = labels
                best_n_iter = n_iter
                if track_history:
                    best_history = history
        
        self.cluster_centers_ = best_centroids
        self.labels_ = best_labels
        self.inertia_ = best_inertia
        self.n_iter_ = best_n_iter
        self._centroid_history = best_history
        
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Predict the closest cluster for each sample.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            New data to predict.
        
        Returns
        -------
        labels : ndarray of shape (n_samples,)
            Predicted cluster index for each sample.
        """
        if self.cluster_centers_ is None:
            raise RuntimeError("Model must be fitted before calling predict()")
        
        X = np.asarray(X, dtype=np.float64)
        labels, _ = self._assign_clusters(X, self.cluster_centers_)
        return labels
    
    def fit_predict(self, X: np.ndarray) -> np.ndarray:
        """
        Fit the model and predict cluster labels.
        
        Convenience method equivalent to calling fit() then predict().
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Training data.
        
        Returns
        -------
        labels : ndarray of shape (n_samples,)
            Cluster labels.
        """
        self.fit(X)
        return self.labels_
    
    def transform(self, X: np.ndarray) -> np.ndarray:
        """
        Transform X to cluster-distance space.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Data to transform.
        
        Returns
        -------
        distances : ndarray of shape (n_samples, n_clusters)
            Distance from each sample to each cluster center.
        """
        if self.cluster_centers_ is None:
            raise RuntimeError("Model must be fitted before calling transform()")
        
        X = np.asarray(X, dtype=np.float64)
        distances_sq = self._compute_distances(X, self.cluster_centers_)
        return np.sqrt(distances_sq)

---

## 3. Training & Optimization

In [ ]:
from sklearn.datasets import make_blobs

# Generate synthetic data with clear cluster structure
X, y_true = make_blobs(
    n_samples=500,
    n_features=2,
    centers=4,
    cluster_std=0.8,
    random_state=42
)

print(f"Dataset shape: {X.shape}")
print(f"Number of true clusters: {len(np.unique(y_true))}")

In [ ]:
# Visualize the original data
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c='steelblue', alpha=0.6, edgecolors='none')
plt.title('Data (Unknown Labels)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', alpha=0.6, edgecolors='none')
plt.title('Data (True Labels - For Reference)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.tight_layout()
plt.show()

In [ ]:
# Train our K-Means implementation
kmeans = KMeans(
    n_clusters=4,
    init='k-means++',
    max_iter=300,
    tol=1e-4,
    n_init=10,
    random_state=42
)

labels = kmeans.fit_predict(X)

print(f"Converged in {kmeans.n_iter_} iterations")
print(f"Inertia (WCSS): {kmeans.inertia_:.2f}")
print(f"\nCluster centers:\n{kmeans.cluster_centers_}")

In [ ]:
# Compare random vs k-means++ initialization
results = {'random': [], 'k-means++': []}

for init_method in ['random', 'k-means++']:
    for seed in range(20):
        km = KMeans(n_clusters=4, init=init_method, n_init=1, random_state=seed)
        km.fit(X)
        results[init_method].append(km.inertia_)

print("Initialization Method Comparison (20 runs each, n_init=1):")
print(f"\nRandom initialization:")
print(f"  Mean inertia: {np.mean(results['random']):.2f}")
print(f"  Std inertia:  {np.std(results['random']):.2f}")
print(f"  Min inertia:  {np.min(results['random']):.2f}")
print(f"  Max inertia:  {np.max(results['random']):.2f}")

print(f"\nK-means++ initialization:")
print(f"  Mean inertia: {np.mean(results['k-means++']):.2f}")
print(f"  Std inertia:  {np.std(results['k-means++']):.2f}")
print(f"  Min inertia:  {np.min(results['k-means++']):.2f}")
print(f"  Max inertia:  {np.max(results['k-means++']):.2f}")

---

## 4. Diagnostics & Evaluation

In [ ]:
def compute_silhouette_score(X: np.ndarray, labels: np.ndarray) -> float:
    """
    Compute the mean silhouette coefficient for all samples.
    
    Parameters
    ----------
    X : ndarray of shape (n_samples, n_features)
        Data points.
    labels : ndarray of shape (n_samples,)
        Cluster labels for each sample.
    
    Returns
    -------
    silhouette : float
        Mean silhouette coefficient.
    """
    n_samples = len(X)
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels)
    
    if n_clusters == 1:
        return 0.0
    
    silhouette_vals = np.zeros(n_samples)
    
    # Compute pairwise distances (squared)
    X_sq = np.sum(X ** 2, axis=1, keepdims=True)
    distances = X_sq + X_sq.T - 2 * (X @ X.T)
    np.maximum(distances, 0, out=distances)
    distances = np.sqrt(distances)
    
    for i in range(n_samples):
        own_cluster = labels[i]
        own_cluster_mask = labels == own_cluster
        own_cluster_size = np.sum(own_cluster_mask)
        
        # a(i) = mean intra-cluster distance
        if own_cluster_size > 1:
            a_i = np.sum(distances[i, own_cluster_mask]) / (own_cluster_size - 1)
        else:
            a_i = 0.0
        
        # b(i) = min mean distance to other clusters
        b_i = np.inf
        for label in unique_labels:
            if label == own_cluster:
                continue
            other_cluster_mask = labels == label
            mean_dist = np.mean(distances[i, other_cluster_mask])
            b_i = min(b_i, mean_dist)
        
        # Silhouette coefficient for sample i
        silhouette_vals[i] = (b_i - a_i) / max(a_i, b_i)
    
    return np.mean(silhouette_vals)


def compute_silhouette_samples(X: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """
    Compute silhouette coefficient for each sample.
    
    Parameters
    ----------
    X : ndarray of shape (n_samples, n_features)
        Data points.
    labels : ndarray of shape (n_samples,)
        Cluster labels for each sample.
    
    Returns
    -------
    silhouette_vals : ndarray of shape (n_samples,)
        Silhouette coefficient for each sample.
    """
    n_samples = len(X)
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels)
    
    if n_clusters == 1:
        return np.zeros(n_samples)
    
    silhouette_vals = np.zeros(n_samples)
    
    # Compute pairwise distances
    X_sq = np.sum(X ** 2, axis=1, keepdims=True)
    distances = X_sq + X_sq.T - 2 * (X @ X.T)
    np.maximum(distances, 0, out=distances)
    distances = np.sqrt(distances)
    
    for i in range(n_samples):
        own_cluster = labels[i]
        own_cluster_mask = labels == own_cluster
        own_cluster_size = np.sum(own_cluster_mask)
        
        if own_cluster_size > 1:
            a_i = np.sum(distances[i, own_cluster_mask]) / (own_cluster_size - 1)
        else:
            a_i = 0.0
        
        b_i = np.inf
        for label in unique_labels:
            if label == own_cluster:
                continue
            other_cluster_mask = labels == label
            mean_dist = np.mean(distances[i, other_cluster_mask])
            b_i = min(b_i, mean_dist)
        
        silhouette_vals[i] = (b_i - a_i) / max(a_i, b_i)
    
    return silhouette_vals

In [ ]:
# Evaluate our clustering
silhouette = compute_silhouette_score(X, labels)
print(f"Evaluation Metrics:")
print(f"  Inertia (WCSS): {kmeans.inertia_:.2f}")
print(f"  Silhouette Score: {silhouette:.4f}")

In [ ]:
# Elbow method: Find optimal K
K_range = range(1, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
    
    if k > 1:
        sil = compute_silhouette_score(X, km.labels_)
        silhouettes.append(sil)
    else:
        silhouettes.append(0)  # Silhouette undefined for k=1

print("K\tInertia\t\tSilhouette")
print("-" * 40)
for k, inertia, sil in zip(K_range, inertias, silhouettes):
    sil_str = f"{sil:.4f}" if k > 1 else "N/A"
    print(f"{k}\t{inertia:.2f}\t\t{sil_str}")

---

## 5. Visualizations

In [ ]:
# Plot 1: Cluster assignments with centroids
plt.figure(figsize=(10, 8))

# Scatter plot of data points colored by cluster
scatter = plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', 
                      alpha=0.6, edgecolors='none', s=50)

# Plot centroids
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            c='red', marker='X', s=200, edgecolors='black', linewidths=2,
            label='Centroids')

plt.colorbar(scatter, label='Cluster')
plt.title('K-Means Clustering Results', fontsize=14)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot 2: Elbow curve and Silhouette score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='r', linestyle='--', label='Optimal K=4')
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia (WCSS)', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=14)
axes[0].set_xticks(list(K_range))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette score plot
axes[1].plot(list(K_range)[1:], silhouettes[1:], 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=4, color='r', linestyle='--', label='Optimal K=4')
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs K', fontsize=14)
axes[1].set_xticks(list(K_range)[1:])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Silhouette plot for detailed cluster analysis
def plot_silhouette(X, labels, ax=None):
    """
    Create a silhouette plot showing the silhouette coefficient for each sample.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    silhouette_vals = compute_silhouette_samples(X, labels)
    n_clusters = len(np.unique(labels))
    
    y_lower = 10
    
    for i in range(n_clusters):
        # Get silhouette values for cluster i
        cluster_silhouette_vals = silhouette_vals[labels == i]
        cluster_silhouette_vals.sort()
        
        size_cluster_i = len(cluster_silhouette_vals)
        y_upper = y_lower + size_cluster_i
        
        color = plt.cm.viridis(i / n_clusters)
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                         0, cluster_silhouette_vals,
                         facecolor=color, edgecolor=color, alpha=0.7)
        
        # Label cluster number
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        
        y_lower = y_upper + 10
    
    ax.axvline(x=np.mean(silhouette_vals), color='red', linestyle='--',
               label=f'Mean: {np.mean(silhouette_vals):.3f}')
    ax.set_xlabel('Silhouette Coefficient')
    ax.set_ylabel('Cluster')
    ax.set_title('Silhouette Plot')
    ax.legend()
    
    return ax

fig, ax = plt.subplots(figsize=(10, 6))
plot_silhouette(X, labels, ax)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Centroid movement visualization (animation concept)
# This shows the progression of centroids during training

if kmeans._centroid_history is not None:
    history = kmeans._centroid_history
    n_steps = min(len(history), 6)  # Show up to 6 steps
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    step_indices = np.linspace(0, len(history) - 1, n_steps, dtype=int)
    
    for idx, step in enumerate(step_indices):
        ax = axes[idx]
        centroids = history[step]
        
        # Assign labels based on current centroids
        distances_sq = np.sum((X[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2, axis=2)
        step_labels = np.argmin(distances_sq, axis=1)
        
        # Plot data points
        ax.scatter(X[:, 0], X[:, 1], c=step_labels, cmap='viridis', 
                   alpha=0.5, edgecolors='none', s=30)
        
        # Plot centroids
        ax.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X', 
                   s=200, edgecolors='black', linewidths=2)
        
        # Draw lines showing centroid movement from previous step
        if step > 0:
            prev_centroids = history[step - 1]
            for k in range(len(centroids)):
                ax.annotate('', xy=centroids[k], xytext=prev_centroids[k],
                           arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
        
        ax.set_title(f'Iteration {step}')
        ax.set_xlabel('Feature 1')
        ax.set_ylabel('Feature 2')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Centroid Movement During K-Means Training', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Centroid history not available.")

In [ ]:
# Plot 5: Decision boundaries (Voronoi diagram)
def plot_decision_boundaries(X, kmeans, ax=None, resolution=200):
    """
    Plot K-Means decision boundaries using a mesh grid.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, resolution),
        np.linspace(y_min, y_max, resolution)
    )
    
    # Predict cluster for each point in mesh
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = kmeans.predict(grid_points)
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundaries
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.5)
    
    # Plot data points
    ax.scatter(X[:, 0], X[:, 1], c=kmeans.labels_, cmap='viridis', 
               edgecolors='black', linewidths=0.5, s=50)
    
    # Plot centroids
    ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
               c='red', marker='X', s=200, edgecolors='black', linewidths=2,
               label='Centroids')
    
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title('K-Means Decision Boundaries')
    ax.legend()
    
    return ax

fig, ax = plt.subplots(figsize=(10, 8))
plot_decision_boundaries(X, kmeans, ax)
plt.tight_layout()
plt.show()

---

## 6. Use Cases & Guidelines

### When to Use K-Means

K-Means is appropriate when:

1. **Clusters are spherical/globular**: K-Means assumes clusters are convex and isotropic (roughly spherical)

2. **Clusters have similar sizes**: The algorithm tends to produce clusters of similar spatial extent

3. **Number of clusters (K) is known or can be estimated**: Elbow method and silhouette analysis can help

4. **Data is numeric and continuous**: K-Means uses Euclidean distance

5. **Computational efficiency is important**: K-Means is very fast compared to hierarchical or density-based methods

**Common applications:**
- Customer segmentation
- Image compression (color quantization)
- Document clustering
- Anomaly detection (points far from any centroid)
- Feature learning (cluster centroids as features)

### When NOT to Use K-Means

K-Means is NOT appropriate when:

1. **Clusters have non-spherical shapes**: Elongated, curved, or irregular clusters

2. **Clusters have varying densities**: Dense clusters adjacent to sparse ones

3. **Clusters have very different sizes**: K-Means may split large clusters or merge small ones

4. **Data contains many outliers**: Outliers significantly affect centroid positions

5. **Number of clusters is unknown and difficult to estimate**

6. **Data is categorical or mixed**: Consider K-modes or K-prototypes instead

**Alternatives for these cases:**
- DBSCAN: For arbitrary shapes and varying densities
- Gaussian Mixture Models: For elliptical clusters
- Hierarchical clustering: When K is unknown
- Spectral clustering: For complex cluster structures

In [ ]:
# Demonstration: K-Means limitations with non-spherical data
from sklearn.datasets import make_moons, make_circles

# Generate challenging datasets
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

# Apply K-Means
km_moons = KMeans(n_clusters=2, random_state=42)
km_circles = KMeans(n_clusters=2, random_state=42)

labels_moons = km_moons.fit_predict(X_moons)
labels_circles = km_circles.fit_predict(X_circles)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Moons - true labels
axes[0, 0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', alpha=0.6)
axes[0, 0].set_title('Moons Dataset (True Labels)')
axes[0, 0].set_xlabel('Feature 1')
axes[0, 0].set_ylabel('Feature 2')

# Moons - K-Means
axes[0, 1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_moons, cmap='viridis', alpha=0.6)
axes[0, 1].scatter(km_moons.cluster_centers_[:, 0], km_moons.cluster_centers_[:, 1],
                   c='red', marker='X', s=200, edgecolors='black')
axes[0, 1].set_title('Moons Dataset (K-Means - FAILS)')
axes[0, 1].set_xlabel('Feature 1')
axes[0, 1].set_ylabel('Feature 2')

# Circles - true labels
axes[1, 0].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', alpha=0.6)
axes[1, 0].set_title('Circles Dataset (True Labels)')
axes[1, 0].set_xlabel('Feature 1')
axes[1, 0].set_ylabel('Feature 2')

# Circles - K-Means
axes[1, 1].scatter(X_circles[:, 0], X_circles[:, 1], c=labels_circles, cmap='viridis', alpha=0.6)
axes[1, 1].scatter(km_circles.cluster_centers_[:, 0], km_circles.cluster_centers_[:, 1],
                   c='red', marker='X', s=200, edgecolors='black')
axes[1, 1].set_title('Circles Dataset (K-Means - FAILS)')
axes[1, 1].set_xlabel('Feature 1')
axes[1, 1].set_ylabel('Feature 2')

plt.suptitle('K-Means Limitations: Non-Spherical Clusters', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### K Selection Strategies

1. **Domain Knowledge**: Often the best approach - do you know how many segments exist?

2. **Elbow Method**: Look for the "elbow" in the inertia curve

3. **Silhouette Score**: Choose K with highest average silhouette

4. **Gap Statistic**: Compare within-cluster dispersion to null reference distribution

5. **Cross-validation**: If using clusters for downstream tasks

### Initialization Importance

- **Always use k-means++**: Significant improvement over random initialization
- **Multiple runs (n_init)**: Default to 10, increase for important applications
- **Reproducibility**: Set random_state for consistent results

---

## 7. Comparison with sklearn

In [ ]:
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.metrics import silhouette_score as sklearn_silhouette_score
import time

In [ ]:
# Create a larger dataset for meaningful comparison
X_large, _ = make_blobs(
    n_samples=5000,
    n_features=10,
    centers=8,
    cluster_std=1.0,
    random_state=42
)

print(f"Dataset shape: {X_large.shape}")

In [ ]:
# Our implementation
our_kmeans = KMeans(n_clusters=8, init='k-means++', n_init=10, random_state=42)

start_time = time.time()
our_labels = our_kmeans.fit_predict(X_large)
our_time = time.time() - start_time

our_silhouette = sklearn_silhouette_score(X_large, our_labels)

print("Our Implementation:")
print(f"  Time: {our_time:.4f} seconds")
print(f"  Inertia: {our_kmeans.inertia_:.2f}")
print(f"  Silhouette Score: {our_silhouette:.4f}")
print(f"  Iterations: {our_kmeans.n_iter_}")

In [ ]:
# sklearn implementation
sklearn_kmeans = SklearnKMeans(n_clusters=8, init='k-means++', n_init=10, random_state=42)

start_time = time.time()
sklearn_labels = sklearn_kmeans.fit_predict(X_large)
sklearn_time = time.time() - start_time

sklearn_silhouette = sklearn_silhouette_score(X_large, sklearn_labels)

print("\nsklearn Implementation:")
print(f"  Time: {sklearn_time:.4f} seconds")
print(f"  Inertia: {sklearn_kmeans.inertia_:.2f}")
print(f"  Silhouette Score: {sklearn_silhouette:.4f}")
print(f"  Iterations: {sklearn_kmeans.n_iter_}")

In [ ]:
# Comparison summary
print("\n" + "="*60)
print("COMPARISON SUMMARY")
print("="*60)
print(f"\n{'Metric':<25} {'Our Impl.':<15} {'sklearn':<15}")
print("-"*55)
print(f"{'Time (seconds)':<25} {our_time:<15.4f} {sklearn_time:<15.4f}")
print(f"{'Inertia':<25} {our_kmeans.inertia_:<15.2f} {sklearn_kmeans.inertia_:<15.2f}")
print(f"{'Silhouette Score':<25} {our_silhouette:<15.4f} {sklearn_silhouette:<15.4f}")
print(f"{'Iterations':<25} {our_kmeans.n_iter_:<15} {sklearn_kmeans.n_iter_:<15}")

print(f"\nSpeed ratio (sklearn/ours): {our_time/sklearn_time:.2f}x")
print(f"Inertia difference: {abs(our_kmeans.inertia_ - sklearn_kmeans.inertia_):.2f}")

In [ ]:
# Compare cluster assignments (accounting for label permutation)
from scipy.optimize import linear_sum_assignment

def compute_assignment_accuracy(labels1, labels2):
    """
    Compute assignment accuracy between two clusterings,
    handling label permutation using Hungarian algorithm.
    """
    n_clusters = max(labels1.max(), labels2.max()) + 1
    
    # Build contingency matrix
    contingency = np.zeros((n_clusters, n_clusters), dtype=np.int64)
    for l1, l2 in zip(labels1, labels2):
        contingency[l1, l2] += 1
    
    # Find optimal assignment
    row_ind, col_ind = linear_sum_assignment(-contingency)
    
    # Compute accuracy
    accuracy = contingency[row_ind, col_ind].sum() / len(labels1)
    return accuracy

accuracy = compute_assignment_accuracy(our_labels, sklearn_labels)
print(f"\nCluster assignment agreement: {accuracy*100:.2f}%")

In [ ]:
# Visual comparison on 2D data
X_2d, _ = make_blobs(n_samples=500, n_features=2, centers=4, 
                      cluster_std=0.8, random_state=42)

# Fit both models
our_km_2d = KMeans(n_clusters=4, random_state=42)
sklearn_km_2d = SklearnKMeans(n_clusters=4, random_state=42)

our_labels_2d = our_km_2d.fit_predict(X_2d)
sklearn_labels_2d = sklearn_km_2d.fit_predict(X_2d)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Our implementation
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=our_labels_2d, cmap='viridis', alpha=0.6)
axes[0].scatter(our_km_2d.cluster_centers_[:, 0], our_km_2d.cluster_centers_[:, 1],
                c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[0].set_title(f'Our Implementation\nInertia: {our_km_2d.inertia_:.2f}')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# sklearn
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=sklearn_labels_2d, cmap='viridis', alpha=0.6)
axes[1].scatter(sklearn_km_2d.cluster_centers_[:, 0], sklearn_km_2d.cluster_centers_[:, 1],
                c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[1].set_title(f'sklearn Implementation\nInertia: {sklearn_km_2d.inertia_:.2f}')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.suptitle('Comparison: Our Implementation vs sklearn', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Key Differences and Notes

| Aspect | Our Implementation | sklearn |
|--------|-------------------|--------|
| Algorithm | Lloyd's algorithm | Lloyd's with optimizations |
| Performance | Pure NumPy | Cython-optimized |
| Memory | Standard | More memory-efficient |
| Features | Basic | Mini-batch, sparse support |

**Why sklearn is faster:**
1. Cython-compiled C extensions
2. Optimized distance computations
3. Memory-efficient chunked operations
4. Parallelization support (n_jobs parameter)

**Our implementation advantages:**
1. Clear, readable code for learning
2. Easy to modify and extend
3. Centroid history tracking for visualization
4. No external dependencies beyond NumPy

---

## Summary

This notebook covered:

1. **Theory**: Lloyd's algorithm, initialization methods (random vs k-means++), convergence criteria, and methods for choosing K

2. **Implementation**: A complete K-Means class from scratch with NumPy, including k-means++ initialization

3. **Training**: Using make_blobs for synthetic data, comparing initialization methods

4. **Evaluation**: Inertia, silhouette score, elbow method

5. **Visualization**: Cluster assignments, decision boundaries, centroid movement, silhouette plots

6. **Guidelines**: When to use K-Means and when to consider alternatives

7. **Comparison**: Benchmarking against sklearn's optimized implementation

**Key Takeaways:**
- K-Means is fast and effective for spherical clusters
- K-means++ initialization significantly improves results
- The elbow method and silhouette score help choose K
- K-Means fails on non-spherical or varying-density clusters
- sklearn's implementation is highly optimized but our pure NumPy version achieves similar results